In [1]:
import pandas as pd
# Veriyi yükleme
df = pd.read_csv("mockito_all_pull_requests_with_update_over_time.csv")

#remove duplicates
#df = df.drop_duplicates(keep='first')

df.drop("pr_title",inplace = True,axis=1)
df.drop("class_names",inplace = True,axis=1)
df.drop("commit_sha",inplace = True,axis=1)
df.drop("commit_message",inplace = True,axis=1)
df.drop("file_path",inplace = True,axis=1)
df.drop("additions",inplace = True,axis=1)
df.drop("deletions",inplace = True,axis=1)

df.drop("created_at",inplace = True,axis=1)
df.drop("closed_at",inplace = True,axis=1)
df.drop("status",inplace = True,axis=1)
df.drop("patch",inplace = True,axis=1)

# Filter rows where 'labels' doesn't contain '|'
df = df[~df['labels'].str.contains('|', regex=False)]

df


,full_repo_name,labels,file_name
0,mockito/mockito,bug,InterceptedInvocation.java
1,mockito/mockito,bug,MockingProgress.java
2,mockito/mockito,bug,MockingProgressImpl.java
3,mockito/mockito,bug,MockCreationListener.java
4,mockito/mockito,bug,StaticMockTest.java
...,...,...,...
72397,mockito/mockito,enhancement,VerificationDataImpl.java
72398,mockito/mockito,enhancement,VerificationData.java
72399,mockito/mockito,enhancement,MatchableInvocation.java
72400,mockito/mockito,enhancement,VerificationData.java


In [2]:
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression # Classification
from sklearn.multioutput import MultiOutputClassifier
from sklearn.metrics import classification_report
from sklearn.metrics import accuracy_score

# Sütun isimlerini kullanarak kodlama işlemleri
le_class = LabelEncoder()
le_repo = LabelEncoder()

# Kodlama işlemleri
df['file_name_encoded'] = le_class.fit_transform(df['file_name'])
df['full_repo_name_encoded'] = le_repo.fit_transform(df['full_repo_name'])

# One-hot encoding ile labels sütununu dönüştürme
df_encoded = pd.get_dummies(df, columns=['labels'])

# Özellikler ve etiketler
X = df_encoded[['file_name_encoded', 'full_repo_name_encoded']]
y = df_encoded.drop(columns=['file_name', 'full_repo_name', 'file_name_encoded', 'full_repo_name_encoded'])
y = y.astype(int) # convert it 0,1     instead of true, false 

# Eğitim ve test setlerine bölme
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42) #random_state: 0,3074,42


In [3]:
# Lojistik regresyon modelini kullanarak çoklu etiket sınıflandırıcı oluşturma
model = MultiOutputClassifier(LogisticRegression())
model.fit(X_train, y_train)

MultiOutputClassifier(estimator=LogisticRegression())

In [4]:
# Örnek tahminler
sample_data = pd.DataFrame({
    'file_name': ['StaticMockTest.java','MockCreationListener.java','MockingProgressImpl.java'], 
    'full_repo_name': ['mockito/mockito','mockito/mockito','mockito/mockito']
})

# Kodlama işlemleri
sample_data['file_name_encoded'] = le_class.transform(sample_data['file_name'])
sample_data['full_repo_name_encoded'] = le_repo.transform(sample_data['full_repo_name'])

# Özellikleri hazırlama
X_sample = sample_data[['file_name_encoded', 'full_repo_name_encoded']]


predictions = model.predict(X_sample)
predictions_df = pd.DataFrame(predictions, columns=y.columns)
#accuracy_logreg = accuracy_score(y_test, predictions)
#print("Logistic Regression Accuracy:", accuracy_logreg)

predictions_df

,labels_bug,labels_enhancement
0,0,1
1,0,1
2,0,1


In [5]:
predictions = model.predict(X_test)
predictions_df = pd.DataFrame(predictions, columns=y.columns)
accuracy_logreg = accuracy_score(y_test, predictions)
print("Logistic Regression Accuracy:", accuracy_logreg)

Logistic Regression Accuracy: 0.9984225747086145
